### Copyright (C) Infineon Technologies AG 2025
 
Copyright (c) 2025, Infineon Technologies AG, or an affiliate of Infineon Technologies AG. All rights reserved.
This software, associated documentation and materials ("Software") is owned by Infineon Technologies AG or one of its affiliates ("Infineon") and is protected by and subject to worldwide patent protection, worldwide copyright laws, and international treaty provisions. Therefore, you may use this Software only as provided in the license agreement accompanying the software package from which you obtained this Software. If no license agreement applies, then any use, reproduction, modification, translation, or compilation of this Software is prohibited without the express written permission of Infineon.

Disclaimer: UNLESS OTHERWISE EXPRESSLY AGREED WITH INFINEON, THIS SOFTWARE IS PROVIDED AS-IS, WITH NO WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING, BUT NOT LIMITED TO, ALL WARRANTIES OF NON-INFRINGEMENT OF THIRD-PARTY RIGHTS AND IMPLIED WARRANTIES SUCH AS WARRANTIES OF FITNESS FOR A SPECIFIC USE/PURPOSE OR MERCHANTABILITY. Infineon reserves the right to make changes to the Software without notice. You are responsible for properly designing, programming, and testing the functionality and safety of your intended application of the Software, as well as complying with any legal requirements related to its use. Infineon does not guarantee that the Software will be free from intrusion, data theft or loss, or other breaches ("Security Breaches"), and Infineon shall have no liability arising out of any Security Breaches. Unless otherwise explicitly approved by Infineon, the Software may not be used in any application where a failure of the Product or any consequences of the use thereof can reasonably be expected to result in personal injury.

# MobileNetV3 for Traffic Object Classification

This notebook adapts an ImageNet1K-pretrained MobileNetV3-small model to classify a selected set of traffic-related objects, such as bikes and cars. It loads the retained class indices, evaluates representative images from the Sautkin ImageNet validation subset, exports the model and input/output tensors, converts the result for embedded deployment, and compares inference performance across supported targets. This is an inference and model-adaptation example; it does not retrain the original MobileNetV3 weights.

## Notebook Structure

1. **Import libraries and helper functions**
2. **Load the ImageNet-pretrained MobileNetV3-small model**
3. **Select traffic-related classes**
4. **Download and inspect validation inputs**
5. **Adapt the classifier to the retained traffic classes**
6. **Generate representative input and output data**
7. **Export the model and test data**
8. **Evaluate predictions and the confusion matrix**
9. **Convert the model for embedded deployment**
10. **Display inference timing and profiling results**
11. **Compare performance across hardware platforms**
12. **References and Generated Artifacts**

## 1. Import libraries and helper functions

Import PyTorch utilities, the MobileNet helpers, repository model helpers, and the conversion client used by the later export and benchmarking stages.

In [ ]:
import sys
import os

from torchsummary import summary
import modelling_helper as mh

parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from _CentralScripts.python_flask_client import CallTools
import _CentralScripts.helper_functions as cs
import _CentralScripts.mobilenet_helper as mo

device = cs.get_device()

## 2. Load the ImageNet-pretrained MobileNetV3-small model

MobileNetV3-small is initialized with the default ImageNet1K pretrained weights [1, 2]. The original model predicts 1,000 ImageNet classes; the following stages retain only classes relevant to this traffic-object example.

In [ ]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

weights = MobileNet_V3_Small_Weights.DEFAULT
model_original = mobilenet_v3_small(weights=weights)

## 3. Select traffic-related classes

`traffic_classes.json` contains the ImageNet class indices retained for this example. Reducing the output space to traffic-related classes lowers the deployment output size and focuses evaluation on the intended use case.

In [ ]:
classes = mh.get_classes_json("traffic_classes.json")

## 4. Download and inspect validation inputs

The helper downloads the Sautkin ImageNet1K validation subset from Kaggle [3] through `kagglehub` and selects representative images for inspection. These images provide inference and validation inputs; they are not a training dataset for this notebook. The selected classes and source images may not represent all traffic scenes, camera conditions, or geographic environments.

In [ ]:
folder_path = mh.get_data()
_, input_size = mh.get_random_input(folder_path, classes, is_plot=True)

## 5. Adapt the classifier to the retained traffic classes

The model is reduced to the indices in `traffic_classes.json`. This preserves the pretrained ImageNet features while removing unrelated output classes. The resulting model is named `traffic_mobilenet` for export and deployment.

In [ ]:
model_name = "traffic_mobilenet"
model = mh.get_model_indices(model_original, classes)
summary(model, input_size, device="cpu")

## 6. Generate representative input and output data

Select one representative image, run inference with the adapted model, and retain the transformed input tensor together with its expected output. These tensors define the input/output contract used to validate conversion and embedded execution.

In [ ]:
input_target, input_size = mh.get_random_input(folder_path, classes)
output_target, predicted_class = mh.get_predicted_class(input_target, model, classes)

## 7. Export the model and test data

Save the adapted PyTorch model, representative input, and expected output. The helper produces the ONNX model and serialized test artifacts consumed by the conversion service.

In [ ]:
cs.save_all(model_name, input_target, output_target, model, origin="torch")

## 8. Evaluate predictions and the confusion matrix

Visualize predictions for several validation images, then calculate a confusion matrix over the selected validation data. These checks measure model behavior separately from the target-specific timing results.

In [ ]:
mo.calculate_predictions_and_plot(model, folder_path, classes)

### Confusion-matrix validation

The confusion matrix summarizes class-level validation behavior for the retained traffic classes. Interpret it together with the source-data limitations described above.

In [ ]:
mh.calculate_confusion_matrix(folder_path, model, device, classes)

## 9. Convert the model for embedded deployment

Prepare the Docker conversion service and convert the exported model for embedded targets. The deployment targets documented by this example are AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4.

In [ ]:
cs.ensure_docker_container()

The conversion service can generate and profile the model for AURIX™ and TRAVEO™ platforms. Select one target for a focused run or use the comparison target for side-by-side results. The available implementation identifiers are `tc3xx`, `tc4dx`, `tc4dx_ppu`, and `arm_m4`; these correspond to AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4 respectively.

In [ ]:
model_folder, onnx_model_file = cs.get_output_paths(model_name)

target = "tc4dx"  # Available targets: "tc3xx", "tc4dx", "tc4dx_ppu", "arm_m4"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
    profile=True,
    tsim=False,  # use qemu simulation backend for tricore targets for faster simulation, expect less accurate timing results
)
tool.convert_model()

## 10. Display inference timing and profiling results

Summarize the selected target's `results.json` timing section, including estimated cycles, instructions, average CPI, latency, and throughput. Profile shares show the contribution of model operators to execution cost and should not be confused with classification quality.

In [ ]:
cs.display_inference_benchmark(model_folder, target)

In [ ]:
cs.plot_profile_shares(model_folder, target)

## 11. Compare performance across hardware platforms

Run the comparison conversion and display timing results across AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4. Use the same exported model and representative input/output artifacts for a meaningful comparison.

In [ ]:
target = "compare"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
    tsim=False,  # use qemu simulation backend for tricore targets for faster simulation, expect less accurate timing results
)
tool.convert_model()

In [ ]:
df = cs.display_platform_comparison(model_folder)

## 12. References and Generated Artifacts

- **Model:** Howard, A. et al. (2019), [Searching for MobileNetV3](https://openaccess.thecvf.com/content_ICCV_2019/html/Howard_Searching_for_MobileNetV3_ICCV_2019_paper.html).
- **Pretraining data:** Russakovsky, O. et al. (2015), [ImageNet Large Scale Visual Recognition Challenge](https://image-net.org/).
- **Validation input:** [Sautkin ImageNet1K validation dataset on Kaggle](https://www.kaggle.com/datasets/sautkin/imagenet1kvalid), downloaded through `kagglehub` and filtered using `traffic_classes.json`. It is used for inference validation rather than model training.
- **Generated model/data:** `traffic_mobilenet` ONNX model, representative transformed input, and expected output data.
- **Deployment artifacts:** conversion source, binaries, logs, timing, and profiling results under `out/traffic_mobilenet/test_traffic_mobilenet/<target>/`.
- **Deployment targets:** AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4.
- **Limitations:** the retained classes and validation images are a narrow subset of ImageNet and do not establish robustness across all traffic environments, sensors, or object appearances.